In [1]:
# Cell 1 — Overrides (primary input)
# Required: "moon" must always be set to a path relative to EXPORTS_ROOT.
# Optional: set any DSO or Planet objectId to pin a specific image instead of auto-selecting.
# objectIds are always lowercase (e.g. "m42", "jupiter", "moon").

image_overrides = {
    "moon": "Lunar/Moon/full/full.jpg",
    "venus": "Planet/Venus/Ven_202602_lapl6_ap38_WS.jpg",
    "mars": "Planet/Mars/Mars_013951_lapl6_ap41_Drizzle15_PI_WS.jpg",
    "jupiter": "Planet/Jupiter/jupiter_grs_io/jupiter_grs_io.jpg",
    "saturn":"Planet/Saturn/saturn_titan/saturn_titan.jpg",
    "ic410":"DSO/IC410/ic410.jpg",
    "ic434":"DSO/IC434/ic4342.jpg",
    "ic1805":"DSO/IC1805/ic1805.jpg",
    "m16":"DSO/M16/m16.jpg",
    "m42":"DSO/M42/m42.jpg", 
    "ngc2264":"DSO/NGC2264/ngc2264.jpg",  
}

In [2]:
# Cell 2 — Imports & config

import json
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps

EXPORTS_ROOT      = Path("/Users/jwatts/Documents/astrophotography/Exports")
OUTPUT_DIR        = Path("mobile")
IMAGES_JSON       = OUTPUT_DIR / "images.json"
DSO_JSON          = OUTPUT_DIR / "dso.json"
MAX_HEIGHT        = 1080
WEBP_QUALITY      = 88
GITHUB_PAGES_BASE = "https://jeffrwatts.github.io/AstroPlannerData/mobile"

IMAGE_EXTENSIONS  = {".jpg", ".jpeg"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR.resolve()}")

Output dir: /Users/jwatts/StudioProjects/git/AstroPlannerData/mobile


In [3]:
# Cell 3 — Discovery

def most_recent_image(folder: Path) -> Path | None:
    """Return the most recently modified image file in folder (non-recursive)."""
    images = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS]
    return max(images, key=lambda f: f.stat().st_mtime) if images else None

def discover_dso() -> dict[str, Path]:
    """DSO/<FolderName>/ → objectId = folder name lowercased."""
    result = {}
    dso_root = EXPORTS_ROOT / "DSO"
    if not dso_root.exists():
        print(f"  WARNING: {dso_root} not found")
        return result
    for folder in sorted(dso_root.iterdir()):
        if not folder.is_dir():
            continue
        img = most_recent_image(folder)
        if img:
            result[folder.name.lower()] = img
        else:
            print(f"  WARNING: no images found in {folder}")
    return result

def discover_planets() -> dict[str, Path]:
    """Planets/<Planet>/<session>/ → objectId = planet folder name lowercased.
    Picks the image from the most recently modified session subfolder."""
    result = {}
    planets_root = EXPORTS_ROOT / "Planet"
    if not planets_root.exists():
        print(f"  WARNING: {planets_root} not found")
        return result
    for planet_folder in sorted(planets_root.iterdir()):
        if not planet_folder.is_dir():
            continue
        session_folders = [d for d in planet_folder.iterdir() if d.is_dir()]
        if not session_folders:
            img = most_recent_image(planet_folder)
        else:
            latest_session = max(session_folders, key=lambda d: d.stat().st_mtime)
            img = most_recent_image(latest_session)
        if img:
            result[planet_folder.name.lower()] = img
        else:
            print(f"  WARNING: no images found under {planet_folder}")
    return result

def build_image_df(overrides: dict[str, str]) -> pd.DataFrame:
    """Merge discovery results, apply overrides, validate moon is present.
    Returns a DataFrame with columns: objectId, src_path, is_override."""
    image_map = {}
    image_map.update(discover_dso())
    image_map.update(discover_planets())

    # Apply overrides (required for moon, optional for others)
    override_set = set()
    for object_id, rel_path in overrides.items():
        image_map[object_id] = EXPORTS_ROOT / rel_path
        override_set.add(object_id)

    if "moon" not in image_map:
        raise ValueError("'moon' is not in image_map. Add it to image_overrides in Cell 1.")

    rows = [
        {"objectId": oid, "src_path": str(path), "is_override": oid in override_set}
        for oid, path in sorted(image_map.items())
    ]
    df = pd.DataFrame(rows)

    # Print summary table
    print(f"{'objectId':<20} {'override?':<10} source")
    print("-" * 80)
    for _, row in df.iterrows():
        print(f"  {row['objectId']:<18} {'yes' if row['is_override'] else 'no':<10} {row['src_path']}")
    print(f"\nTotal: {len(df)} objects")
    return df

image_df = build_image_df(image_overrides)

objectId             override?  source
--------------------------------------------------------------------------------
  barnard68          no         /Users/jwatts/Documents/astrophotography/Exports/DSO/barnard68/barnard68.jpg
  barnard72          no         /Users/jwatts/Documents/astrophotography/Exports/DSO/barnard72/barnard72.jpg
  ic1805             yes        /Users/jwatts/Documents/astrophotography/Exports/DSO/IC1805/ic1805.jpg
  ic410              yes        /Users/jwatts/Documents/astrophotography/Exports/DSO/IC410/ic410.jpg
  ic434              yes        /Users/jwatts/Documents/astrophotography/Exports/DSO/IC434/ic4342.jpg
  ic443              no         /Users/jwatts/Documents/astrophotography/Exports/DSO/IC443/IC443.jpg
  ic5146             no         /Users/jwatts/Documents/astrophotography/Exports/DSO/IC5146/IC5146.jpg
  jupiter            yes        /Users/jwatts/Documents/astrophotography/Exports/Planet/Jupiter/jupiter_grs_io/jupiter_grs_io.jpg
  m1                 n

In [4]:
# Cell 3b — Join with DSO catalog & add thumb columns

dso_df = pd.read_json(DSO_JSON)

# Left join: keep all image objects; DSO fields will be NaN for planets/moon
images_df = image_df.merge(dso_df[["objectId", "displayName", "ra", "dec", "constellation"]],
                           on="objectId", how="left")

# Fill missing displayName (planets/moon) with capitalized objectId
images_df["displayName"] = images_df.apply(
    lambda row: row["objectId"].capitalize() if pd.isna(row["displayName"]) else row["displayName"],
    axis=1
)

# Add thumbnail columns (null for now)
images_df["thumbX"]   = None
images_df["thumbY"]   = None
images_df["thumbDim"] = None

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
images_df[["objectId", "displayName", "ra", "dec", "constellation", "thumbX", "thumbY", "thumbDim"]]

,objectId,displayName,ra,dec,constellation,thumbX,thumbY,thumbDim
0,barnard68,Barnard 68 (Ophiuchus Globule),17.377278,-23.826111,NaN,None,None,None
1,barnard72,Barnard 72 (Snake Nebula),17.394167,-23.695000,NaN,None,None,None
2,ic1805,Heart Nebula,2.544864,61.456889,Cassiopeia,None,None,None
3,ic410,Tadpoles Nebula,5.378333,33.366667,Auriga,None,None,None
4,ic434,Horsehead Nebula,5.683578,-2.453778,Orion,None,None,None
5,ic443,IC 443 (Jellyfish Nebula),6.277058,22.531667,Gemini,None,None,None
6,ic5146,Cocoon Nebula,21.891322,47.266917,Cygnus,None,None,None
7,jupiter,Jupiter,NaN,NaN,NaN,None,None,None
8,m1,Crab Nebula,5.575547,22.014472,Taurus,None,None,None
9,m101,Pinwheel Galaxy,14.053483,54.348944,Ursa Major,None,None,None


In [5]:
# Cell 4 — Image processing

for _, row in images_df.iterrows():
    object_id = row["objectId"]
    src = Path(row["src_path"])
    dst = OUTPUT_DIR / f"{object_id}.webp"

    with Image.open(src) as img:
        img = ImageOps.exif_transpose(img)

        w, h = img.size
        if h > MAX_HEIGHT:
            scale = MAX_HEIGHT / h
            new_size = (round(w * scale), MAX_HEIGHT)
            img = img.resize(new_size, Image.LANCZOS)
            print(f"  {object_id}: {w}x{h} → {img.size[0]}x{img.size[1]}  →  {dst.name}")
        else:
            print(f"  {object_id}: {w}x{h} (no resize needed)  →  {dst.name}")

        img.save(dst, format="WEBP", quality=WEBP_QUALITY)

print(f"\nDone. {len(images_df)} images written to {OUTPUT_DIR.resolve()}")

  barnard68: 4137x2816 → 1587x1080  →  barnard68.webp
  barnard72: 4135x2815 → 1586x1080  →  barnard72.webp
  ic1805: 6240x4162 → 1619x1080  →  ic1805.webp
  ic410: 4052x2674 → 1637x1080  →  ic410.webp
  ic434: 2800x2800 → 1080x1080  →  ic434.webp
  ic443: 6238x4167 → 1617x1080  →  ic443.webp
  ic5146: 4136x2811 → 1589x1080  →  ic5146.webp
  jupiter: 1024x768 (no resize needed)  →  jupiter.webp
  m1: 3907x2585 → 1632x1080  →  m1.webp
  m101: 4129x2811 → 1586x1080  →  m101.webp
  m13: 4138x2820 → 1585x1080  →  m13.webp
  m16: 4134x2815 → 1586x1080  →  m16.webp
  m20: 4118x2791 → 1593x1080  →  m20.webp
  m27: 4126x2814 → 1584x1080  →  m27.webp
  m3: 4134x2818 → 1584x1080  →  m3.webp
  m31: 6242x4173 → 1615x1080  →  m31.webp
  m42: 4130x2800 → 1593x1080  →  m42.webp
  m45: 4415x2948 → 1617x1080  →  m45.webp
  m51: 4120x2800 → 1589x1080  →  m51.webp
  m8: 4122x2812 → 1583x1080  →  m8.webp
  m83: 4125x2814 → 1583x1080  →  m83.webp
  mars: 900x643 (no resize needed)  →  mars.webp
  moon: 304

In [ ]:
# Cell 5 — Generate & serialize images.json

OUTPUT_COLS = ["objectId", "displayName", "ra", "dec", "constellation", "thumbX", "thumbY", "thumbDim"]

out_df = images_df[OUTPUT_COLS].copy()
out_df["url"] = out_df["objectId"].apply(lambda oid: f"{GITHUB_PAGES_BASE}/{oid}.webp")

records = json.loads(out_df.to_json(orient="records"))
(OUTPUT_DIR / "images.json").write_text(json.dumps(records, indent=2))
print(f"Wrote {len(records)} records to {(OUTPUT_DIR / 'images.json').resolve()}")
out_df